In [3]:
import pandas as pd

In [4]:
df_main = pd.read_excel('data_royxat.xlsx')


In [11]:
df_main[['name', 'cert_num']].duplicated().sum()

np.int64(428)

In [5]:
df_sub = pd.read_excel('content.xlsx')

In [ ]:
df_sub

In [ ]:
# Find first occurrence of duplicates
duplicate_mask = df_main.duplicated(subset=['name', 'cert_num'], keep='first')

# Create DataFrame with only first instance of duplicates
df_duplicates_first = df_main[duplicate_mask].copy()

# Print summary
print("\nDuplicate entries (first instance only):")
print(f"Number of duplicates: {len(df_duplicates_first)}")
print("\nFirst few duplicate entries:")
print(df_duplicates_first.head())

In [10]:
df_duplicates_first.to_excel('duplicate_entries_first.xlsx', index=False)

In [ ]:
# 1. Check for exact duplicate pairs in df_sub
print("Duplicate pairs in sub dataset:")
dup_sub = df_sub[['name', 'cert_num']].duplicated().sum()
print(f"Number of duplicates in sub: {dup_sub}")

# 2. Check for exact duplicate pairs in df_main
print("\nDuplicate pairs in main dataset:")
dup_main = df_main[['name', 'cert_num']].duplicated().sum()
print(f"Number of duplicates in main: {dup_main}")

# 3. Analyze the null values in main dataset
print("\nRows with null names in main dataset:")
print(df_main[df_main['name'].isnull()][['name', 'cert_num', 'dob']].head())

# 4. Check for any spacing/case inconsistencies
df_main['name_stripped'] = df_main['name'].str.strip().str.upper()
df_sub['name_stripped'] = df_sub['name'].str.strip().str.upper()

# Perform merge with cleaned names
df_result_clean = df_main.merge(df_sub[['name_stripped', 'cert_num']], 
                               how='left', 
                               indicator=True,
                               left_on=['name_stripped', 'cert_num'],
                               right_on=['name_stripped', 'cert_num'])

df_result_clean = df_result_clean[df_result_clean['_merge'] == 'left_only'].drop(['_merge', 'name_stripped'], axis=1)

print("\nAfter cleaning:")
print(f"Cleaned result rows: {df_result_clean.shape[0]:,}")


In [9]:
# Clean the names while preserving NaN values
df_main_clean = df_main.copy(deep=True)
df_main_clean['name'] = df_main_clean['name'].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
df_sub['name'] = df_sub['name'].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)

# Identify and separate duplicates
duplicate_mask = df_main_clean.duplicated(subset=['name', 'cert_num'], keep=False)
duplicates_df = df_main_clean[duplicate_mask].copy()
non_duplicates_df = df_main_clean[~duplicate_mask].copy()

# Process non-duplicates
df_sub = df_sub.copy(deep=True)
df_sub['to_remove'] = True

merged_df = non_duplicates_df.merge(
    df_sub[['name', 'cert_num', 'to_remove']],
    on=['name', 'cert_num'],
    how='left',
    validate='many_to_one'
)

# Filter and get final non-duplicates
filtered_df = merged_df[merged_df['to_remove'] != True].drop(columns=['to_remove'])

print("Statistics:")
print(f"Original duplicates found: {len(duplicates_df):,}")
print(f"Non-duplicates processed: {len(filtered_df):,}")


Statistics:
Original duplicates found: 858
Non-duplicates processed: 45,154


In [2]:
filtered_df = pd.read_excel('filtered_data.xlsx')

In [ ]:
filtered_df

In [4]:
import os
import math
import pandas as pd

# Create directory if it doesn't exist
output_dir = 'split_excel_files'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Calculate number of files needed
rows_per_file = 1000
total_files = math.ceil(len(filtered_df) / rows_per_file)

# Split and save files
for i in range(total_files):
    start_idx = i * rows_per_file
    end_idx = min((i + 1) * rows_per_file, len(filtered_df))
    
    # Get chunk of data
    chunk_df = filtered_df.iloc[start_idx:end_idx]
    
    # Create filename with padding zeros for proper sorting
    filename = f'filtered_data_{str(i+1).zfill(3)}.xlsx'
    filepath = os.path.join(output_dir, filename)
    
    # Save to Excel
    chunk_df.to_excel(filepath, index=False)
    
print(f"Split complete: {total_files} files created in '{output_dir}' directory")
print(f"Total rows processed: {len(filtered_df)}")

Split complete: 46 files created in 'split_excel_files' directory
Total rows processed: 45154


In [6]:
# Clean the names while preserving NaN values
df_main_clean = df_main.copy(deep=True)
df_main_clean['name'] = df_main_clean['name'].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
df_sub['name'] = df_sub['name'].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)

# Store duplicates separately
duplicate_mask = df_main_clean.duplicated(subset=['name', 'cert_num'], keep=False)
duplicates_df = df_main_clean[duplicate_mask].copy()
non_duplicates_df = df_main_clean[~duplicate_mask].copy()

# Add marker column to df_sub
df_sub = df_sub.copy(deep=True)
df_sub['to_remove'] = True

# Merge non-duplicates with df_sub
merged_df = non_duplicates_df.merge(
    df_sub[['name', 'cert_num', 'to_remove']],
    on=['name', 'cert_num'],
    how='left',
    validate='many_to_one'
)

# Filter out rows marked for removal
filtered_df = merged_df[merged_df['to_remove'] != True].drop(columns=['to_remove'])

# Add duplicates back
# df_final = pd.concat([filtered_df, duplicates_df], ignore_index=True)

# Debugging output
print("Intermediate Debugging:")
print(f"Original duplicates found: {len(duplicates_df):,}")
print(f"Rows in merged_df before filtering: {merged_df.shape[0]:,}")
print(f"Rows marked for removal: {merged_df['to_remove'].notnull().sum():,}")
# print(f"Rows retained after filtering: {df_final.shape[0]:,}")

print("\nFinal Dataset Statistics:")
print(f"Original main dataset: {df_main.shape[0]:,} rows")
print(f"Sub dataset: {df_sub.shape[0]:,} rows")
# print(f"Final cleaned result: {df_final.shape[0]:,} rows")

# # Verify row count
# expected_rows = df_main.shape[0] - df_sub.shape[0]
# if df_final.shape[0] != expected_rows:
#     print(f"\nRow counts:")
#     print(f"Expected: {expected_rows}")
#     print(f"Actual: {df_final.shape[0]}")
#     print(f"Difference: {abs(expected_rows - df_final.shape[0])}")

Intermediate Debugging:
Original duplicates found: 858
Rows in merged_df before filtering: 54,063
Rows marked for removal: 8,909

Final Dataset Statistics:
Original main dataset: 54,921 rows
Sub dataset: 9,440 rows


In [ ]:
filtered_df

In [ ]:
duplicates_df

In [10]:
df_duplicates_first.to_excel('duplicates_first.xlsx', index=False)